In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# House Prices: Multiple Linear Regression

Data: Kaggle "House Prices - Advanced Regression Techniques" (Ames, Iowa)
Path used here: `/kaggle/input/house-prices-advanced-regression-techniques/train.csv`

We will:
1. Explore the target distribution
2. Inspect correlations in a readable way
3. Start with a small, intuitive feature set
4. Train a multiple linear regression
5. Evaluate with R², adjusted R², RMSE
6. Inspect residuals
7. Use statsmodels to get p-values and confidence intervals
8. Look at how we might use feature-selection.

## Import and set up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm

# Optional: scikit-learn feature selection helpers
from sklearn.feature_selection import SelectKBest, f_regression, RFE

# Reproducibility and environment info
import sys, sklearn, statsmodels
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("statsmodels:", statsmodels.__version__)

# Plot defaults
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False


### Load and check the shape.

In [ ]:
# Kaggle default path
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
df.shape

### Get the first 5 rows (head) of the table.

In [ ]:
df.head()

### List all the columns

In [ ]:
df.info()

## Exploratory Data Analysis (EDA)
### Visualize the sales price distribution

In [ ]:
plt.figure()
plt.hist(df["SalePrice"], bins=40, edgecolor="black")
plt.title("SalePrice distribution")
plt.xlabel("SalePrice")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

df["SalePrice"].describe()

### Get the top 20 correlations

In [ ]:
# Work on numeric features only for correlations
num_df = df.select_dtypes(include=[np.number]).copy()

# Basic correlation matrix
corr = num_df.corr()

# Sorted correlations with the target
target_corr = corr["SalePrice"].sort_values(ascending=False)
target_corr.head(20)

### Visualize them as a heatmap.

In [ ]:
# Vertical heatmap for target correlations (clean and readable)
import seaborn as sns

plt.figure(figsize=(8, 12))
sns.heatmap(target_corr.to_frame(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation of features with SalePrice")
plt.tight_layout()
plt.show()

### Create a correlation matrix

In [ ]:
top10 = target_corr.index[:10]
plt.figure(figsize=(10, 8))
sns.heatmap(num_df[top10].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Top 10 correlated features heatmap")
plt.tight_layout()
plt.show()

### Create a pairwise plot

This will help us check for collinearity.

In [ ]:
sns.pairplot(num_df[top10])
plt.suptitle("Pairplot of Top 10 Features (Collinearity Check)", y=1.02)
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = df[top10].fillna(0)  # use your chosen features
X_vif = sm.add_constant(X_vif)  # add intercept for statsmodels

vif_data = pd.DataFrame()
vif_data["feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i)
                   for i in range(X_vif.shape[1])]
vif_data

In [ ]:
# Let's choose the top 5 features.
features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF"
]

X = df[features].copy()
y = df["SalePrice"].copy()

# Simple missing value handling for this first pass
X = X.fillna(0)
y = y.fillna(y.median())

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Fit linear regression
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)

print("R²:", round(r2, 4))
print("Adjusted R²:", round(adj_r2, 4))
print("RMSE:", round(rmse, 2))

pd.DataFrame({"feature": X.columns, "coefficient": lr.coef_}).sort_values(
    "coefficient", ascending=False
)

### Plot of residuals

In [ ]:
residuals = y_test - y_pred

plt.figure()
plt.scatter(y_pred, residuals, alpha=0.8)
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals (y - ŷ)")
plt.tight_layout()
plt.show()

plt.figure()
plt.hist(residuals, bins=30, edgecolor="black")
plt.title("Distribution of residuals")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### Summary

In [ ]:
X_sm = sm.add_constant(X)  # adds intercept term
ols = sm.OLS(y, X_sm).fit()
print(ols.summary())

#### Create table for coefficients
Below we're just creating an easier table to read. You can see the p-values, lower and upper bounds and the b values to add to your equation.

In [ ]:
pvals = ols.pvalues.rename("p_value")
conf_int = ols.conf_int(alpha=0.05)
conf_int.columns = ["ci_lower_95", "ci_upper_95"]

summary_tbl = pd.concat([ols.params.rename("coef"), pvals, conf_int], axis=1)
summary_tbl

## Feature Selection: Forward, Backward, (and Stepwise)

We apply two classical feature selection methods:

- **Forward selection:** start empty, add features that improve the model the most
- **Backward elimination:** start with all features, remove the least useful one each step
- **Stepwise (bidirectional):** combination of forward + backward, dynamically adding or removing features

We use `sklearns`’s `SequentialFeatureSelector`, which doesn't support stepwise. That's why we're only going to demonstrate forward and backwards selection.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

# Numeric features only, drop the target
X_all = num_df.drop(columns=["SalePrice"]).fillna(0)
y_all = df["SalePrice"]

# Prefilter to top 20 features by absolute correlation with SalePrice
top20 = num_df.corr()["SalePrice"].abs().sort_values(ascending=False).index[1:21]
X_pool = X_all[top20].copy()

est = LinearRegression()

### Forward Selection

Here you can see below that we're taking the top 5 features, using r-squared (r2) as the scoring, and going in a forward direction.

In [ ]:
sfs_fwd = SequentialFeatureSelector(
    est,
    n_features_to_select=5,
    direction="forward",
    scoring="r2"
)
sfs_fwd.fit(X_pool, y_all)

fwd_features = X_pool.columns[sfs_fwd.get_support()].tolist()
print("Forward-selected features:", fwd_features)

### Backward Selection
Again, looking below, we're choosing the top 5 features and using r-squared (r2).

In [ ]:
sfs_bwd = SequentialFeatureSelector(
    est,
    n_features_to_select=5,
    direction="backward",
    scoring="r2"
)
sfs_bwd.fit(X_pool, y_all)

bwd_features = X_pool.columns[sfs_bwd.get_support()].tolist()
print("Backward-selected features:", bwd_features)

### Compare to original selection
Let's generate the residuals and then get the results of the model.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

chosen = fwd_features

X = X_pool[chosen]
y = y_all

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

lr = LinearRegression().fit(X_train, y_train)
y_pred = lr.predict(X_test)

r2 = r2_score(y_test, y_pred)
adj_r2 = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print("Chosen features:", chosen)
print("Holdout R²:", round(r2, 4))
print("Adjusted R²:", round(adj_r2, 4))
print("RMSE:", round(rmse, 2))

# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(7,4))
plt.scatter(y_pred, residuals, alpha=0.8)
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residuals (y - ŷ)")
plt.tight_layout()
plt.show()

# Histogram of residuals
plt.figure(figsize=(7,4))
plt.hist(residuals, bins=30, edgecolor="black")
plt.title("Distribution of residuals")
plt.xlabel("Residual")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm

X_sm = sm.add_constant(X)   # add intercept
ols = sm.OLS(y, X_sm).fit()
print(ols.summary())

# Tidy table if you want it in a dataframe
pvals = ols.pvalues.rename("p_value")
conf_int = ols.conf_int(alpha=0.05)
conf_int.columns = ["ci_lower_95", "ci_upper_95"]
summary_tbl = pd.concat([ols.params.rename("coef"), pvals, conf_int], axis=1)
summary_tbl

## Summary
- We explored the target distribution and correlations in a readable way.
- We trained a simple multiple linear regression with five intuitive features.
- We reported R², adjusted R², and RMSE, and examined residuals.
- Using statsmodels, we reviewed p-values and 95 percent confidence intervals.
- We looked at feature selection.